# AIaaS — ASR Throughput Benchmark (faster-whisper)

Portable throughput benchmark for the **speech-to-text** workload, using
**faster-whisper** (CTranslate2) — the efficient Whisper runtime the strategy doc
names for the ASR proxy tier.

Reports:
- **RTF** (real-time factor = processing-time ÷ audio-seconds; lower is better)
- **speedup** (audio-seconds ÷ processing-time; "× real-time")
- **audio-seconds processed per wall-second** (throughput)
- **GPU memory used**

> Tier: *portable proxy*. Uses **synthetic audio** for timing, so it measures
> throughput, **not** transcription accuracy (no WER). For credible numbers use
> MLPerf **whisper** (LibriSpeech + WER gate) in the `inference` fork.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "faster-whisper", "numpy", "pandas"], check=False)
import faster_whisper
print("faster-whisper", faster_whisper.__version__)


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def gpu_mem_used_mb():
    v = smi("memory.used")
    try:
        return float(v[0]) if v else None
    except Exception:
        return None

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."
_cc = torch.cuda.get_device_capability(0)

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "compute_capability": f"{_cc[0]}.{_cc[1]}",
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
VRAM-tiered Whisper size. `float16` compute on GPU.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"
MODEL = "large-v3" if TIER == "large" else "small"   # faster-whisper model size
COMPUTE_TYPE = "float16"        # int8_float16 is even lighter; float16 is the common GPU default
BEAM_SIZE = 1                   # greedy = throughput-oriented

CLIP_SECONDS = 30               # whisper processes in 30s windows
NUM_CLIPS = 10                  # clips transcribed (after warmup)
SAMPLE_RATE = 16000             # whisper expects 16 kHz mono

OUT_DIR = "asr_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"VRAM={VRAM} GB -> TIER={TIER}, MODEL=whisper-{MODEL}, compute={COMPUTE_TYPE}, "
      f"clip={CLIP_SECONDS}s x{NUM_CLIPS}")


## 4. Synthetic audio
A band-limited noise + tone mix at 16 kHz — content is meaningless (this is a
*timing* run), but the decoder does real work over `CLIP_SECONDS` of audio.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_clip(seconds, sr=SAMPLE_RATE):
    n = int(seconds * sr)
    t = np.arange(n) / sr
    tone = 0.1 * np.sin(2 * np.pi * 220 * t).astype(np.float32)
    noise = 0.02 * rng.standard_normal(n).astype(np.float32)
    return (tone + noise).astype(np.float32)

CLIPS = [make_clip(CLIP_SECONDS) for _ in range(NUM_CLIPS)]
TOTAL_AUDIO_S = CLIP_SECONDS * NUM_CLIPS
print(f"{NUM_CLIPS} clips, {TOTAL_AUDIO_S}s total audio")


## 5. Load model + transcribe sweep

In [ ]:
import time
from faster_whisper import WhisperModel

mem_before = gpu_mem_used_mb()
t = time.time()
model = WhisperModel(MODEL, device="cuda", compute_type=COMPUTE_TYPE)
load_s = time.time() - t
mem_after_load = gpu_mem_used_mb()

def transcribe(clip):
    # segments is a generator; consume it to force the work.
    segments, info = model.transcribe(clip, beam_size=BEAM_SIZE)
    return sum(1 for _ in segments)

# warmup
transcribe(CLIPS[0])
mem_peak = gpu_mem_used_mb()

t0 = time.time()
for clip in CLIPS:
    transcribe(clip)
proc_s = time.time() - t0

RESULT = {
    "load_s": round(load_s, 3),
    "proc_s_total": round(proc_s, 3),
    "audio_s_total": TOTAL_AUDIO_S,
    "rtf": round(proc_s / TOTAL_AUDIO_S, 4),
    "speedup_x_realtime": round(TOTAL_AUDIO_S / proc_s, 2),
    "audio_s_per_wall_s": round(TOTAL_AUDIO_S / proc_s, 2),
    "gpu_mem_used_mb": (round(mem_peak - mem_before, 1)
                        if (mem_peak and mem_before) else mem_after_load),
}
print(json.dumps(RESULT, indent=2))


## 6. Results — save + summarize

In [ ]:
import pandas as pd
NORM = {
    "schema": "asr-bench/1.0",
    "env": ENV, "model": f"whisper-{MODEL}", "compute_type": COMPUTE_TYPE,
    "beam_size": BEAM_SIZE, "clip_seconds": CLIP_SECONDS, "num_clips": NUM_CLIPS,
    "result": RESULT,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"asr_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(f"\n==== ASR SUMMARY ====\n{ENV['gpu_name']} | whisper-{MODEL} | {COMPUTE_TYPE}")
df = pd.DataFrame([RESULT])
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))


## 7. Reading the result
**speedup ×real-time** is the headline: e.g. 20× means one GPU can keep up with
~20 concurrent real-time audio streams (capacity). **RTF < 1** means faster than
real-time. For accuracy-gated numbers, run MLPerf whisper on LibriSpeech.